In [1]:
# ------------------------------------------------------------------------
# STEP 0: IMPORTS
# We import everything up front so it's obvious what this script depends on.
# ------------------------------------------------------------------------
import os
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # write plots to files instead of trying to pop up a GUI window
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    average_precision_score,
    precision_recall_curve,
)
import joblib

warnings.filterwarnings("ignore")  # keep console output focused on our own logs
sns.set_theme(style="whitegrid")

# ------------------------------------------------------------------------
# STEP 1: CONFIG
# Everything you're likely to want to change lives here, in one place.
# ------------------------------------------------------------------------
CSV_PATH = r"C:\Users\STR\OneDrive\Desktop\fraud_detection_pipline\core-api\model-training\dataset\dataset.csv"  # <-- point this at your real 1M-row file
OUTPUT_DIR = r"C:\Users\STR\OneDrive\Desktop\fraud_detection_pipline\core-api\model-training\outputs"
OUTPUT_DIR_MODEL = r"C:\Users\STR\OneDrive\Desktop\fraud_detection_pipline\core-api\app\models"
TARGET_COL = "is_fraud"
ID_COL_CANDIDATES = ["transaction_id", "ransaction_id", "trans_id", "id"]  # we auto-detect whichever exists
LEAKY_COLS = ["fraud_type"]  # explained in STEP 4 — this column must NOT be used as a feature
RANDOM_STATE = 42
TEST_SIZE = 0.20  # 20% held out for final evaluation

os.makedirs(OUTPUT_DIR, exist_ok=True)


def savefig(name):
    """Small helper so every plot gets saved consistently and then closed
    (closing prevents matplotlib from silently accumulating memory over
    many plots, which matters when we're doing this repeatedly)."""
    path = os.path.join(OUTPUT_DIR, name)
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.close()
    print(f"  saved plot -> {path}")


# ------------------------------------------------------------------------
# STEP 2: LOAD DATA
# For 1M rows with ~12 columns, a plain pd.read_csv is fine memory-wise
# on most machines, but we still specify dtypes where we can to keep the
# memory footprint down and loading fast.
# ------------------------------------------------------------------------
print("STEP 2: Loading data...")
df = pd.read_csv(CSV_PATH)
print(f"  loaded {len(df):,} rows and {df.shape[1]} columns")

# Normalize the boolean target column: it may arrive as True/False, 1/0,
# or the strings "True"/"False" depending on how the CSV was written.
df[TARGET_COL] = df[TARGET_COL].astype(str).str.lower().map(
    {"true": 1, "1": 1, "false": 0, "0": 0}
).astype(int)

# ------------------------------------------------------------------------
# STEP 3: INITIAL EXPLORATION (EDA)
# Before touching the model, understand what you're working with:
# class balance, missing values, and rough feature distributions.
# ------------------------------------------------------------------------
print("\nSTEP 3: Exploratory data analysis...")

print("\n--- dtypes ---")
print(df.dtypes)

print("\n--- missing values per column ---")
print(df.isna().sum())

fraud_counts = df[TARGET_COL].value_counts()
fraud_rate = df[TARGET_COL].mean()
print(f"\n--- class balance ---\n{fraud_counts}\nFraud rate: {fraud_rate:.4%}")

# Plot 1: class balance (this is the single most important chart in a
# fraud problem — it tells you immediately that this is a HIGHLY imbalanced
# classification task, which drives every modeling choice below).
plt.figure(figsize=(5, 4))
sns.countplot(x=df[TARGET_COL].map({0: "Legit", 1: "Fraud"}))
plt.title(f"Class Balance (fraud rate = {fraud_rate:.3%})")
plt.xlabel("")
savefig("01_class_balance.png")

# Plot 2: fraud_type breakdown (only meaningful among fraud rows).
if "fraud_type" in df.columns:
    plt.figure(figsize=(7, 4))
    df.loc[df[TARGET_COL] == 1, "fraud_type"].value_counts().plot(kind="bar")
    plt.title("Fraud Sub-types (within fraud cases only)")
    plt.ylabel("count")
    savefig("02_fraud_type_breakdown.png")

# Plot 3: numeric feature distributions split by fraud/legit.
# log1p is used for amount-like columns because transaction amounts and
# volumes are typically extremely right-skewed (a few huge values dominate
# a raw histogram and hide the shape of the bulk of the data).
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != TARGET_COL]

n_plots = len(numeric_cols)
n_cols_grid = 3
n_rows_grid = int(np.ceil(n_plots / n_cols_grid))
fig, axes = plt.subplots(n_rows_grid, n_cols_grid, figsize=(15, 4 * n_rows_grid))
axes = axes.flatten()
for i, col in enumerate(numeric_cols):
    ax = axes[i]
    data_to_plot = df[[col, TARGET_COL]].copy()
    sns.kdeplot(data=data_to_plot, x=col, hue=TARGET_COL, ax=ax, common_norm=False, warn_singular=False)
    ax.set_title(col)
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])
savefig("03_feature_distributions_by_class.png")

# Plot 4: correlation heatmap among numeric features (helps spot redundant
# / highly correlated features, which can hurt linear models like
# Logistic Regression more than tree-based models).
plt.figure(figsize=(9, 7))
corr = df[numeric_cols + [TARGET_COL]].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Heatmap")
savefig("04_correlation_heatmap.png")

# ------------------------------------------------------------------------
# STEP 4: FEATURE ENGINEERING & CLEANING
# ------------------------------------------------------------------------
print("\nSTEP 4: Feature engineering...")

work = df.copy()

# 4a. Drop the ID column — it's a unique identifier per row, so a model
# could only "memorize" it, never generalize from it.
id_col = next((c for c in ID_COL_CANDIDATES if c in work.columns), None)
if id_col:
    work = work.drop(columns=[id_col])
    print(f"  dropped ID column: {id_col}")

# 4b. Drop LEAKY columns. `fraud_type` is only populated when is_fraud is
# True — in other words, its very presence/absence tells the model the
# answer directly. Any column that is only known AFTER a transaction is
# confirmed fraudulent (rather than at the moment you'd need to score it)
# must be excluded, or you'll get a model that looks amazing in testing
# and is useless in production. We keep it aside for reporting only.
fraud_type_series = work["fraud_type"].copy() if "fraud_type" in work.columns else None
for col in LEAKY_COLS:
    if col in work.columns:
        work = work.drop(columns=[col])
        print(f"  dropped leaky column: {col} (only known after the fact)")

# 4c. Handle the sentinel value 9999 in days_since_last_trans.
# Looking at the sample data, 9999 clearly means "no prior transaction"
# rather than a literal 9999 days. We convert it into two honest signals:
#   - a binary flag `is_new_account_no_history`
#   - the numeric column capped so it doesn't distort scaling/statistics
if "days_since_last_trans" in work.columns:
    work["is_new_account_no_history"] = (work["days_since_last_trans"] >= 9999).astype(int)
    # replace the sentinel with the max "real" value observed, so the
    # magnitude doesn't dwarf every other feature once scaled
    real_max = work.loc[work["days_since_last_trans"] < 9999, "days_since_last_trans"].max()
    real_max = 0 if pd.isna(real_max) else real_max
    work.loc[work["days_since_last_trans"] >= 9999, "days_since_last_trans"] = real_max
    print("  engineered is_new_account_no_history flag and capped the 9999 sentinel")

# 4d. Log-transform heavily right-skewed monetary/volume columns.
# np.log1p (log(1+x)) is used instead of log(x) because these columns can
# legitimately contain 0, and log(0) is undefined.
skewed_cols = ["trans_amount", "sender_volume_last_1h"]
for col in skewed_cols:
    if col in work.columns:
        work[f"{col}_log"] = np.log1p(work[col].clip(lower=0))

# 4e. Simple ratio feature: how "bursty" is this sender right now relative
# to their typical activity. Domain-informed features like this often help
# tree models less than linear models, but cost little to compute either way.
if {"sender_txn_count_last_1h", "sender_volume_last_1h"}.issubset(work.columns):
    work["avg_amt_per_txn_last_1h"] = work["sender_volume_last_1h"] / work["sender_txn_count_last_1h"].replace(0, np.nan)
    work["avg_amt_per_txn_last_1h"] = work["avg_amt_per_txn_last_1h"].fillna(0)

print(f"  final feature set has {work.shape[1] - 1} columns (excluding target)")

# ------------------------------------------------------------------------
# STEP 5: TRAIN / TEST SPLIT
# stratify=y keeps the same fraud rate in both the train and test sets —
# critical here since fraud is rare and a random split could otherwise
# leave the test set with almost no fraud examples.
# ------------------------------------------------------------------------
print("\nSTEP 5: Train/test split...")

X = work.drop(columns=[TARGET_COL])
y = work[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
print(f"  train: {X_train.shape[0]:,} rows | test: {X_test.shape[0]:,} rows")
print(f"  train fraud rate: {y_train.mean():.4%} | test fraud rate: {y_test.mean():.4%}")

# ------------------------------------------------------------------------
# STEP 6: PREPROCESSING PIPELINE
# We wrap imputation + scaling inside an sklearn Pipeline/ColumnTransformer
# so that (a) statistics like the mean/std used for scaling are learned
# ONLY from the training set (avoiding data leakage from test into train),
# and (b) the exact same transformation is trivially reapplied to any new
# data at prediction time.
# ------------------------------------------------------------------------
print("\nSTEP 6: Building preprocessing pipeline...")

numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="median")),  # median is robust to outliers
                ("scaler", StandardScaler()),
            ]),
            numeric_features,
        )
    ],
    remainder="drop",  # any stray non-numeric column is dropped rather than silently mis-handled
)

# ------------------------------------------------------------------------
# STEP 7: HANDLE CLASS IMBALANCE
# Fraud is rare (see Plot 1). If we train naively, the model can get
# ~98%+ "accuracy" just by predicting "not fraud" every time — which is
# worthless. We counter this with class_weight="balanced", which tells
# each model to penalize mistakes on the minority (fraud) class more
# heavily, roughly proportional to how rare it is.
# (If you later install imbalanced-learn, SMOTE oversampling is another
# common alternative — see the OPTIONAL note near the bottom.)
# ------------------------------------------------------------------------
print("\nSTEP 7: Configuring models with class-imbalance handling...")

models = {
    "logistic_regression": LogisticRegression(
        class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE
    ),
    "random_forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=12,
        class_weight="balanced",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    ),
    "hist_gradient_boosting": HistGradientBoostingClassifier(
        max_iter=300,
        learning_rate=0.08,
        max_depth=8,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),
}

# ------------------------------------------------------------------------
# STEP 8: TRAIN + EVALUATE EACH MODEL
# For fraud detection, plain accuracy is misleading (see Step 7). Instead
# we focus on:
#   - Precision/Recall for the fraud class specifically
#   - ROC-AUC (ranking quality across all thresholds)
#   - PR-AUC / Average Precision (more informative than ROC-AUC when the
#     positive class is rare, which is exactly our situation)
# ------------------------------------------------------------------------
print("\nSTEP 8: Training and evaluating models...\n")

results = {}
fitted_pipelines = {}

for name, model in models.items():
    print(f"--- {name} ---")
    pipe = Pipeline(steps=[("preprocessor", preprocessor), ("model", model)])
    pipe.fit(X_train, y_train)

    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]

    roc_auc = roc_auc_score(y_test, y_proba)
    pr_auc = average_precision_score(y_test, y_proba)
    report = classification_report(y_test, y_pred, target_names=["legit", "fraud"], output_dict=True)

    print(classification_report(y_test, y_pred, target_names=["legit", "fraud"]))
    print(f"ROC-AUC: {roc_auc:.4f} | PR-AUC (avg precision): {pr_auc:.4f}\n")

    results[name] = {
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "report": report,
    }
    fitted_pipelines[name] = pipe

    # Confusion matrix plot per model
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(4.5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["legit", "fraud"], yticklabels=["legit", "fraud"])
    plt.title(f"Confusion Matrix - {name}")
    plt.ylabel("Actual")
    plt.xlabel("Predicted")
    savefig(f"05_confusion_matrix_{name}.png")

# ------------------------------------------------------------------------
# STEP 9: COMPARE MODELS (ROC and PR curves on one plot each)
# ------------------------------------------------------------------------
print("STEP 9: Comparing models visually...")

plt.figure(figsize=(6, 5))
for name, pipe in fitted_pipelines.items():
    y_proba = pipe.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    plt.plot(fpr, tpr, label=f"{name} (AUC={results[name]['roc_auc']:.3f})")
plt.plot([0, 1], [0, 1], "k--", alpha=0.4, label="random guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend()
savefig("06_roc_curve_comparison.png")

plt.figure(figsize=(6, 5))
for name, pipe in fitted_pipelines.items():
    y_proba = pipe.predict_proba(X_test)[:, 1]
    precision, recall, _ = precision_recall_curve(y_test, y_proba)
    plt.plot(recall, precision, label=f"{name} (AP={results[name]['pr_auc']:.3f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve Comparison (more informative for imbalanced data)")
plt.legend()
savefig("07_precision_recall_curve_comparison.png")

# ------------------------------------------------------------------------
# STEP 10: PICK THE BEST MODEL
# We select by PR-AUC rather than ROC-AUC, since PR-AUC is the more
# reliable summary metric when the positive class (fraud) is rare.
# ------------------------------------------------------------------------
best_name = max(results, key=lambda k: results[k]["pr_auc"])
best_pipe = fitted_pipelines[best_name]
print(f"\nSTEP 10: Best model by PR-AUC = {best_name} (PR-AUC={results[best_name]['pr_auc']:.4f})")

# ------------------------------------------------------------------------
# STEP 11: FEATURE IMPORTANCE (for the best model, if tree-based)
# Understanding WHICH signals drive fraud predictions is often as
# valuable to a fraud team as the predictions themselves.
# ------------------------------------------------------------------------
print("\nSTEP 11: Feature importance...")

best_model_step = best_pipe.named_steps["model"]
if hasattr(best_model_step, "feature_importances_"):
    importances = pd.Series(best_model_step.feature_importances_, index=numeric_features)
    importances = importances.sort_values(ascending=False)

    plt.figure(figsize=(8, 5))
    importances.head(15).plot(kind="barh")
    plt.gca().invert_yaxis()
    plt.title(f"Top Feature Importances - {best_name}")
    savefig("08_feature_importance.png")
    print(importances.head(10))
else:
    print(f"  {best_name} has no native feature_importances_ (e.g. Logistic Regression) — "
          f"consider inspecting model.coef_ instead for linear models.")

# ------------------------------------------------------------------------
# STEP 12: THRESHOLD TUNING (business-driven, optional but recommended)
# By default, predict() uses a 0.5 probability threshold. In fraud, you
# usually want to CHOOSE the threshold based on a business trade-off:
# e.g. "we're willing to accept X% false positives to catch Y% more fraud."
# This block scans thresholds and reports recall/precision at each, so a
# fraud team can pick the operating point that matches their risk appetite.
# ------------------------------------------------------------------------
print("\nSTEP 12: Threshold tuning for the best model...")

y_proba_best = best_pipe.predict_proba(X_test)[:, 1]
precision_arr, recall_arr, thresholds_arr = precision_recall_curve(y_test, y_proba_best)

threshold_table = pd.DataFrame({
    "threshold": np.concatenate([thresholds_arr, [1.0]]),
    "precision": precision_arr,
    "recall": recall_arr,
})
# show a handful of representative rows rather than every single threshold
sampled = threshold_table.iloc[:: max(1, len(threshold_table) // 10)]
print(sampled.to_string(index=False))
threshold_table.to_csv(os.path.join(OUTPUT_DIR, "threshold_table.csv"), index=False)
print(f"  full threshold table saved -> {OUTPUT_DIR}/threshold_table.csv")

# ------------------------------------------------------------------------
# STEP 13: SAVE THE FINAL MODEL + METRICS
# We persist the entire Pipeline (preprocessing + model together), so
# scoring new transactions later is a single pipe.predict_proba(new_df)
# call — no need to remember which scaler or imputer was used.
# ------------------------------------------------------------------------
print("\nSTEP 13: Saving the trained model and metrics report...")

model_path = os.path.join(OUTPUT_DIR, f"best_model_{best_name}.joblib")
model_path = os.path.join(OUTPUT_DIR_MODEL, f"fraud_model.pkl")
joblib.dump(best_pipe, model_path)
print(f"  model saved -> {model_path}")

metrics_summary = {name: {"roc_auc": r["roc_auc"], "pr_auc": r["pr_auc"]} for name, r in results.items()}
metrics_path = os.path.join(OUTPUT_DIR, "metrics_summary.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_summary, f, indent=2)
print(f"  metrics summary saved -> {metrics_path}")

print("\nDONE. All plots, the trained model, and metrics are in:", OUTPUT_DIR)

# ------------------------------------------------------------------------
# OPTIONAL EXTENSIONS (not run here, shown for reference)
# ------------------------------------------------------------------------
# 1) XGBoost instead of / alongside HistGradientBoostingClassifier:
#
#     from xgboost import XGBClassifier
#     n_pos = y_train.sum()
#     n_neg = len(y_train) - n_pos
#     xgb_model = XGBClassifier(
#         n_estimators=400, max_depth=6, learning_rate=0.05,
#         scale_pos_weight=n_neg / n_pos,   # xgboost's equivalent of class_weight
#         eval_metric="aucpr", random_state=RANDOM_STATE,
#     )
#
# 2) SMOTE oversampling instead of class_weight (requires: pip install imbalanced-learn):
#
#     from imblearn.over_sampling import SMOTE
#     from imblearn.pipeline import Pipeline as ImbPipeline
#     pipe = ImbPipeline(steps=[
#         ("preprocessor", preprocessor),
#         ("smote", SMOTE(random_state=RANDOM_STATE)),
#         ("model", RandomForestClassifier(random_state=RANDOM_STATE)),
#     ])
#
# 3) Scoring brand-new, unlabeled transactions later:
#
#     loaded_pipe = joblib.load(model_path)
#     new_transactions = pd.read_csv("new_transactions.csv")
#     # apply the same Step 4 feature engineering to new_transactions first!
#     fraud_probabilities = loaded_pipe.predict_proba(new_transactions)[:, 1]
# ------------------------------------------------------------------------

STEP 2: Loading data...
  loaded 1,000,002 rows and 12 columns

STEP 3: Exploratory data analysis...

--- dtypes ---
transaction_id                    str
trans_amount                  float64
age_hours_open_acc            float64
is_fraud                        int64
fraud_type                        str
receiver_txn_count_last_3d    float64
unique_senders_last_3d          int64
multi_same_amt_count_2d       float64
sender_txn_count_last_1h      float64
sender_volume_last_1h         float64
days_since_last_trans         float64
geo_speed_kmh                 float64
dtype: object

--- missing values per column ---
transaction_id                     0
trans_amount                       0
age_hours_open_acc                 0
is_fraud                           0
fraud_type                    700002
receiver_txn_count_last_3d         0
unique_senders_last_3d             0
multi_same_amt_count_2d            0
sender_txn_count_last_1h           0
sender_volume_last_1h              0
days_sin

In [2]:
# Print accuracy for every trained algorithm.
print("\nAccuracy for all algorithms:")
for algorithm_name, algorithm_results in results.items():
    print(f"{algorithm_name}: {algorithm_results['report']['accuracy']:.4%}")


Accuracy for all algorithms:
logistic_regression: 98.8230%
random_forest: 99.0915%
hist_gradient_boosting: 99.2265%


In [3]:
# ------------------------------------------------------------------------
# STEP 14: TEST THE TRAINED MODEL WITH DUMMY DATA
# This uses the same feature engineering as Step 4 before scoring.
# ------------------------------------------------------------------------
print("\nSTEP 14: Testing the trained model with dummy transactions...")

dummy_transactions = pd.DataFrame([
    {
        "trans_amount": 42.50,
        "age_hours_open_acc": 8760,
        "receiver_txn_count_last_3d": 2,
        "unique_senders_last_3d": 2,
        "multi_same_amt_count_2d": 0,
        "sender_txn_count_last_1h": 1,
        "sender_volume_last_1h": 42.50,
        "days_since_last_trans": 14,
        "geo_speed_kmh": 3,
    },
    {
        "trans_amount": 1250000.00,
        "age_hours_open_acc": 2,
        "receiver_txn_count_last_3d": 0,
        "unique_senders_last_3d": 78,
        "multi_same_amt_count_2d": 0,
        "sender_txn_count_last_1h": 1,
        "sender_volume_last_1h": 1250000.00,
        "days_since_last_trans": 9999,
        "geo_speed_kmh": 0,
    },
    {
        "trans_amount": 850.00,
        "age_hours_open_acc": 240,
        "receiver_txn_count_last_3d": 12,
        "unique_senders_last_3d": 4,
        "multi_same_amt_count_2d": 1,
        "sender_txn_count_last_1h": 4,
        "sender_volume_last_1h": 2400.00,
        "days_since_last_trans": 1,
        "geo_speed_kmh": 25,
    },
    {
        "trans_amount": 50000.00,
        "age_hours_open_acc": 12,
        "receiver_txn_count_last_3d": 1,
        "unique_senders_last_3d": 50,
        "multi_same_amt_count_2d": 0,
        "sender_txn_count_last_1h": 8,
        "sender_volume_last_1h": 400000.00,
        "days_since_last_trans": 9999,
        "geo_speed_kmh": 900,
    },
])

dummy_transactions["is_new_account_no_history"] = (
    dummy_transactions["days_since_last_trans"] >= 9999
).astype(int)
real_days = dummy_transactions.loc[
    dummy_transactions["days_since_last_trans"] < 9999,
    "days_since_last_trans",
].max()
dummy_transactions.loc[
    dummy_transactions["days_since_last_trans"] >= 9999,
    "days_since_last_trans",
] = real_days

for col in ["trans_amount", "sender_volume_last_1h"]:
    dummy_transactions[f"{col}_log"] = np.log1p(dummy_transactions[col].clip(lower=0))
dummy_transactions["avg_amt_per_txn_last_1h"] = (
    dummy_transactions["sender_volume_last_1h"] /
    dummy_transactions["sender_txn_count_last_1h"].replace(0, np.nan)
).fillna(0)

# Train a separate fraud-type classifier. fraud_type is excluded from the
# binary model because it is only known after a transaction is labeled fraud.
fraud_rows = y == 1
fraud_type_features = X.loc[fraud_rows]
fraud_type_labels = fraud_type_series.loc[fraud_rows]
fraud_type_train, fraud_type_test, fraud_label_train, fraud_label_test = train_test_split(
    fraud_type_features,
    fraud_type_labels,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=fraud_type_labels,
)

fraud_type_pipe = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=300,
        max_depth=12,
        class_weight="balanced",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )),
])
fraud_type_pipe.fit(fraud_type_train, fraud_label_train)
fraud_type_test_predictions = fraud_type_pipe.predict(fraud_type_test)
print("\nFraud-type classifier report:")
print(classification_report(fraud_label_test, fraud_type_test_predictions, zero_division=0))

# Score the dummy transactions with both models.
dummy_predictions = best_pipe.predict(dummy_transactions)
dummy_probabilities = best_pipe.predict_proba(dummy_transactions)[:, 1]
dummy_type_predictions = fraud_type_pipe.predict(dummy_transactions)
dummy_results = dummy_transactions[["trans_amount", "age_hours_open_acc"]].copy()
dummy_results["fraud_probability"] = dummy_probabilities.round(4)
dummy_results["predicted_label"] = np.where(dummy_predictions == 1, "FRAUD", "LEGIT")
dummy_results["predicted_fraud_type"] = np.where(
    dummy_predictions == 1,
    dummy_type_predictions,
    "NOT_APPLICABLE",
)
print("\nDummy transaction predictions:")
print(dummy_results.to_string(index=False))
# ------------------------------------------------------------------------


STEP 14: Testing the trained model with dummy transactions...

Fraud-type classifier report:
                           precision    recall  f1-score   support

 IMMEDIATE_LARGE_TRANSFER       1.00      1.00      1.00     10000
            LOCATION_JUMP       0.99      0.99      0.99     10000
MANY_TO_ONE_CONSOLIDATION       0.99      0.99      0.99     10000
     MULTIPLE_SAME_AMOUNT       1.00      1.00      1.00     10000
           SLEEP_AND_WAKE       1.00      1.00      1.00     10000
           VELOCITY_SPIKE       0.99      0.99      0.99     10000

                 accuracy                           1.00     60000
                macro avg       1.00      1.00      1.00     60000
             weighted avg       1.00      1.00      1.00     60000


Dummy transaction predictions:
 trans_amount  age_hours_open_acc  fraud_probability predicted_label     predicted_fraud_type
         42.5                8760                0.0           LEGIT           NOT_APPLICABLE
    1250000.0

In [4]:
# ------------------------------------------------------------------------
# STEP 15: SAVE THE FRAUD-TYPE MODEL FOR THE API
# Run Step 14 first so fraud_type_pipe has been trained.
# ------------------------------------------------------------------------
fraud_type_model_path = os.path.join(
    OUTPUT_DIR_MODEL,
    "fraud_type_model.pkl",
)
os.makedirs(OUTPUT_DIR_MODEL, exist_ok=True)
joblib.dump(fraud_type_pipe, fraud_type_model_path)
print(f"Fraud-type model saved -> {fraud_type_model_path}")
# ------------------------------------------------------------------------

Fraud-type model saved -> C:\Users\STR\OneDrive\Desktop\fraud_detection_pipline\core-api\app\models\fraud_type_model.pkl
